# Diabetic kidney disease risk modelling

Exercise notebook for the lesson **[Diabetic kidney disease risk modelling](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-kidney-disease-risk)** in the *Applied case studies: diabetes & metabolic health* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** In diabetes, the risk of kidney failure depends above all on two tests: eGFR, which measures kidney function, and urine albumin-to-creatinine ratio (ACR), which measures kidney damage. KDIGO combines them into a colour-coded 'heat map' of risk categories, and validated equations such as the Kidney Failure Risk Equation turn them into an individual 2- or 5-year risk. On synthetic data, a continuous model using the same tests ranked patients slightly better than the categories, and sparse categories gave unreliable estimates.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Rebuilding the heat map

A synthetic cohort of 20,000 adults with diabetes and the observed 5-year kidney failure rate (%) by category.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(12)
n = 20000
df = pd.DataFrame({
    "age": rng.normal(62, 11, n).clip(30, 90).round(),
    "egfr": rng.gamma(9, 7.5, n).clip(8, 130).round(),          # mL/min/1.73m2
    "acr": np.exp(rng.normal(3.2, 1.6, n)).clip(2, 3000).round(), # urine albumin-to-creatinine, mg/g
})
# Synthetic 5-year risk of kidney failure: higher with lower eGFR and higher albuminuria
logit = -2.6 - 0.075 * (df.egfr - 45) + 0.55 * (np.log(df.acr) - np.log(30)) - 0.01 * (df.age - 62)
df["kidney_failure_5y"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))

df["G"] = pd.cut(df.egfr, [0, 15, 30, 45, 60, 90, 999], right=False,
                 labels=["G5", "G4", "G3b", "G3a", "G2", "G1"])
df["A"] = pd.cut(df.acr, [0, 30, 300, 1e9], right=False, labels=["A1", "A2", "A3"])
heat = df.pivot_table(index="G", columns="A", values="kidney_failure_5y", aggfunc="mean", observed=False)
print((heat.loc[["G1", "G2", "G3a", "G3b", "G4", "G5"]] * 100).round(1).to_string())

# A      A1    A2    A3
# G
# G1    0.0   0.2   0.5
# G2    0.3   2.1   4.4
# G3a   1.7   6.7  18.2
# G3b   7.1  16.5  42.3
# G4   19.7  39.4  52.2
# G5    0.0  50.0   NaN

### Sparse categories

Continuing from the previous block: how many people fall in each cell?

In [ ]:
counts = df.pivot_table(index="G", columns="A", values="kidney_failure_5y", aggfunc="size", observed=False)
print(counts.loc[["G1", "G2", "G3a", "G3b", "G4", "G5"]].to_string())

# A      A1    A2   A3
# G
# G1   1710  1247  192
# G2   4916  3457  477
# G3a  2708  1992  330
# G3b  1456  1014  137
# G4    193   142   23
# G5      4     2    0

### Categories versus a continuous model

Continuing from the previous blocks: heat-map cell risk against a logistic model of eGFR, log ACR and age.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df.kidney_failure_5y)
# 1. Heat-map cell: use the observed risk of each G x A cell in the training data
cell_risk = train.groupby(["G", "A"], observed=False).kidney_failure_5y.mean()
p_cell = test.set_index(["G", "A"]).index.map(cell_risk).to_numpy(dtype=float)
# 2. Continuous model using the same information
features = lambda d: np.column_stack([d.egfr, np.log(d.acr), d.age])
model = LogisticRegression(max_iter=1000).fit(features(train), train.kidney_failure_5y)
p_model = model.predict_proba(features(test))[:, 1]
print(f"KDIGO heat-map categories: AUROC {roc_auc_score(test.kidney_failure_5y, p_cell):.3f}")
print(f"Continuous eGFR, log ACR, age: AUROC {roc_auc_score(test.kidney_failure_5y, p_model):.3f}")

# KDIGO heat-map categories: AUROC 0.849
# Continuous eGFR, log ACR, age: AUROC 0.867

## Exercise

**Goal:** Model kidney risk in diabetes with categories, continuous predictors and survival methods.

**Data:** Synthetic cohort generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the heat map and cell counts, and add bootstrap confidence intervals to each cell's risk.

In [ ]:
# Your code here


### Step 2

Add follow-up time and censoring, and refit the model as a Cox model from the survival analysis lesson.

In [ ]:
# Your code here


### Step 3

Add a competing risk of death and compare kidney failure risk estimates with and without it.

In [ ]:
# Your code here


### Step 4

Check calibration of the continuous model by decile of predicted risk.

In [ ]:
# Your code here


### Step 5

Write a short explanation of a patient's 5-year risk that a person with diabetes could understand.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-kidney-disease-risk](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-kidney-disease-risk). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.